# Copyright 2026 Google LLC. All Rights Reserved.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-deepmind/atomic_concept_edits/blob/main/colabs/ace_word_count_demo.ipynb)

Licensed under the Apache License, Version 2.0 (the "License");

In [ ]:
# Copyright 2026 Google LLC. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# ACE (Atomic Concept Edits) - Prompt mutation and Model steering

In [ ]:
# @title Imports


from dotenv import load_dotenv
load_dotenv()
import json
from atomic_concept_edits.data.config import DatasetConfig
import os
import random
from atomic_concept_edits import ace_exploration
from atomic_concept_edits import autoraters
from atomic_concept_edits import constitution_optimizer
from atomic_concept_edits import data as datasets
from atomic_concept_edits import interface
from atomic_concept_edits import models
from atomic_concept_edits import mutation_samplers
from atomic_concept_edits import prompts
from atomic_concept_edits.util import eval as eval_util
import matplotlib.pyplot as plt
import pandas as pd

# Sample mutations for any given prompt using - ACEMutationSampler

The mutations are sampled by providing a guidance or objective i.e., what is the goal you are trying to achieve. Some examples are:

*   "Decrease the alignment between the prompt and the generated image."
*   "Make sure the model provides the wrong answer to the mathematical problem"

In [ ]:
objective = 'Force the model to adhere to a word count constraint'

In [ ]:
# Engine LLM - provide your Gemini API key
llm_config = models.GeminiModelConfig(
    model='gemini-3-flash-preview',
    api_key=os.environ.get('GEMINI_API_KEY'),  # Replace with your API key
)

ace_mutation_sampler_config = mutation_samplers.ACEMutationSamplerConfig(
    llm_config=llm_config,
    objective=objective,
    constitution='',
)

mutation_sampler = mutation_samplers.ACEMutationSampler(
    ace_mutation_sampler_config
)

In [ ]:
prompt = 'A cat sitting on a couch'
mutation_sampler.sample(prompt=prompt, num_samples=5)

# Exploration - Generate lots of ACEs on a dataset of initial prompts

In this section, we run the exploration pipeline to generate a dataset of Atomic Concept Edits (ACEs) that satisfy the objective. We need to configure:
1.  **Dataset**: The source of initial prompts (e.g., LIMA).
2.  **Target Model**: The model we want to steer (e.g., Gemini 2.0 Flash).
3.  **Autorater**: To judge if the objective is satisfied.
4.  **Exploration Config**: To control the search depth and width.

In [ ]:
# Sample initial prompts
prompt_list = [
    'Write a poem about a cat',
    'Write an story about a dog',
    'Plan a trip to Tokyo',
    'How do I replace my car tire?',
]


# Target Model - that we want to steer
target_model_config = models.GeminiModelConfig(
    model='gemini-3-flash-preview',
    api_key=os.environ.get('GEMINI_API_KEY'),  # Replace with your API key
)

# Autorater to judge if the objective is satisfied or not - in this case, a word count autorater
word_count_autorater_config = autoraters.WordCountAutoraterConfig(
    target_word_count=50, word_count_slope=1.0
)

save_path = '/tmp/ace_exploration'

dataset_config = DatasetConfig(dataset_name='custom', num_prompts=len(prompt_list), prompt_list=prompt_list)

config = ace_exploration.ACEExplorationConfig(
    mutation_sampler_config=ace_mutation_sampler_config,
    target_model_config=target_model_config,
    autorater_config=word_count_autorater_config,
    dataset_config=dataset_config,
    sample_size_at_depth=(2, 2, 2),
    num_responses_per_prompt=1,
    min_score=0,  # target autorater score should be exactly 0
    max_score=0,
    save_path=save_path,
)

exploration = ace_exploration.ACEExploration(config)
run_id = exploration.run()

In [ ]:
df = pd.read_csv(f'{save_path}/{run_id}/exploration_data.csv')
df

In [ ]:
# @title Generated data that satisfies the objective

print(
    'Number of generated prompts that satisfy the objective,'
    f' {len(df[df["objective_satisfied"] == True])}'
)

In [ ]:
# @title Display a sequence of ACEs with model responses

# select a random sequence of ACEs that satisfies the objective
row_index = random.choice(
    df[
        (df['objective_satisfied'] == True) & (df['target_model_responses'].notna())
    ].index
)
ace_sequence = []
while True:
  print()
  ace_sequence.append((
      df.at[row_index, 'prompt'],
      df.at[row_index, 'objective_satisfied'],
      df.at[row_index, 'target_model_responses'],
  ))
  if pd.isna(df.at[row_index, 'parent_id']):
    break
  row_index = df[df['prompt_id'] == df.at[row_index, 'parent_id']].index[0]

print(f'Objective: {objective}')
for seq in ace_sequence[::-1]:
  result = random.choice(json.loads(seq[2]))
  print(seq[0], 'Objective satisfied according to autorater?: ', seq[1])
  print('Response:')
  print(result)

# Learn a constitution of insights that provides reliable strategies that satisfy the objective

We do this by using automatic prompt optimization (like TextGrad or AlphaEvolve). Internally, we build a constitution and use it as a surrogate classifier to predict whether a given input (prompt, ace) satisfies the objective or not (by using the autorater scores as guidance).

In the cells below, we:
1.  Configure the `ConstitutionOptimizer`.
2.  Run the optimizer to find the best constitution.
3.  Plot training progress.

In [ ]:
constitution_optimizer_config = (
    constitution_optimizer.ConstitutionOptimizerConfig(
        save_path=save_path,
        run_id=run_id,
        epochs=2,
        engine_llm_config=llm_config,
        objective=objective,
        initial_constitution='',
        batch_size=10,
        initial_num_strategies=5,
        final_num_strategies=10,
        initial_change_percentage=100,
        final_change_percentage=10,
    )
)


optimizer = constitution_optimizer.ConstitutionOptimizer(
    config=constitution_optimizer_config
)

In [ ]:
best_constitution_on_test_set = optimizer.run_optimizer()

In [ ]:
print(best_constitution_on_test_set)

In [ ]:
# @title View all generated constitutions and train, test, val splits and accuracy results at - {save_path}/{run_id}/constitution/

(
    latest_constitution,
    best_constitution_on_test_set,
    current_epoch,
    train_data_with_results,
    val_data_with_results,
    test_data_with_results,
    all_train_losses,
    all_val_losses,
    all_test_losses,
) = optimizer.load_checkpoint()

In [ ]:
print(best_constitution_on_test_set)

In [ ]:
plt.plot(all_train_losses, label='train')
plt.plot(all_val_losses, label='val')
plt.plot(all_test_losses, label='test')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss of constitution as a surrogate predictor')
plt.show()